# Weerdata AWS_STATION

- We halen de data op via de URL (netwerk) en geven de nodige parameters mee. 
  - de URL is te zien bij het netwerk
- Dit script dient elke maand te laden zodat data automatisch wordt geladen in de database voor als er changes zijn gebeurt die maand met de stations (elke laatste dag van de maand om 23:59:59)

In [ ]:
import requests
import pandas as pd
import io
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

import sys

ROOT = Path().resolve().parents[1] 
sys.path.append(str(ROOT))

from data_gathering.dimLocation.initializer import getLocationKey
from DWH.connection.connect import loadIN



In [2]:
load_dotenv() 

notebook_dir = Path.cwd()

base_data_path = (notebook_dir.parent.parent / "data").resolve()

telpaal_path = base_data_path / "telpalen" / "telpalen_locatie_data.csv"
zipcode_path = base_data_path / "zipcodes_num_nl_2025.xls"

In [3]:
BASE_URL = "https://opendata.meteo.be/service/ows"
GROQ_API_KEY = os.getenv("groq_API")
ZIPCODE_FILE = zipcode_path
df_zip = pd.read_excel(ZIPCODE_FILE)

In [4]:
df_zip.head()

,Postcode,Plaatsnaam,Deelgemeente,Hoofdgemeente,Provincie
0,612,Sinterklaas,NaN,Sinterklaas,NaN
1,1000,Brussel,Neen,BRUSSEL,BRUSSEL
2,1005,Verenigde Vergadering van de Gemeenschappelijke,NaN,Verenigde Vergadering van de Gemeenschappelijke,NaN
3,1006,Raad van de Vlaamse Gemeenschapscommissie,NaN,Raad van de Vlaamse Gemeenschapscommissie,NaN
4,1007,Assemblée de la Commission Communautaire Franç...,NaN,Assemblée de la Commission Communautaire Franç...,NaN


In [5]:
llm = ChatGroq(
    temperature=0, 
    model_name="llama-3.3-70b-versatile", 
    groq_api_key=GROQ_API_KEY)

In [6]:
def get_vlaanderen_data(lat, lon):
    url = f"https://geo.api.vlaanderen.be/geolocation/v4/Location?latlon={lat},{lon}"
    try:
        r = requests.get(url, timeout=5)
        res = r.json().get('LocationResult', [])
        if res:
            return str(res[0].get('Zipcode'))
    except:
        pass
    return None

In [ ]:
def download_aws_stations():
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typenames": "aws:aws_station",
        "outputformat": "text/csv"
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()

        # csv inlezen in DataFrame
        df = pd.read_csv(io.StringIO(response.text))
        print(df.head())
        
        # Geodata parsen
        df["latitude"] = df["the_geom"].str.replace("POINT (", "", regex=False).str.replace(")", "", regex=False).str.split().str[0].astype(float)
        df["longitude"] = df["the_geom"].str.replace("POINT (", "", regex=False).str.replace(")", "", regex=False).str.split().str[1].astype(float)
        df['point'] = df['the_geom'].str.replace("POINT ", "")
        
        for index, row in df.iterrows():
            naam = str(row['name'])
            lat, lon = row['latitude'], row['longitude']
            found = False

            # Via Vlaanderen API
            pc = get_vlaanderen_data(lat, lon) 
            
            print(naam, pc)
            if pc != None:
                print("postcode gevonden in Vlaanderen")
                match_excel = df_zip[df_zip['Postcode'] == int(pc)]
                if not match_excel.empty:
                    found = True 
                    
            
            print(found)
            
            # AI Fallback
            if not found:
                print(f"bezig met AI voor {naam}")
                print(lat,lon)
                prompt = (f"Coördinaten: (latitude: {lat}, longitude: {lon}). "
                            "Taak: Geef uitsluitend de 4-cijferige Belgische Waalse postcode van deze locatie. "
                            "Output: Alleen het getal, geen tekst, geen zinnen.")
                try:
                    ai_pc_raw = llm.invoke(prompt).content.strip()
                    pc = ''.join(filter(str.isdigit, ai_pc_raw)) 
                    print(f"gevonden postcode door AI is: {pc}")
                    if pc:
                        match_excel = df_zip[df_zip['Postcode'] == int(pc)].head(1)
                        if not match_excel.empty:
                            found = True
                except Exception as e:
                    print(f"Fout bij AI lookup voor {naam}: {e}")
            if found:
                province = match_excel.iloc[0]['Provincie'].title()
                mainMunicipality = match_excel.iloc[0]['Hoofdgemeente'].title()
                municipality = match_excel.iloc[0]['Plaatsnaam'].title()
                df.at[index, "LocationKey"] = getLocationKey(postalCode=pc, MainMunicipality=mainMunicipality, Municipality=municipality, Province=province)
        # Datum toevoegen (SnapshotDate)
        df["snapshot_date"] = pd.Timestamp.now().normalize().date()
        
        # Onnodige kolommen verwijderen
        cols_to_drop = ['FID', 'date_begin', 'date_end', 'the_geom']
        df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
        
        # Hernoemen naar PascalCase conform SQL tabel
        df.rename(columns={
            "code": "WeatherStationID",
            "name": "Name",
            "altitude": "Altitude",
            "longitude": "Longitude",
            "latitude": "Latitude",
            "province": "Province",
            "point": "Point",
            "snapshot_date": "SnapshotDate"
        }, inplace=True)

        # Forceer juiste datatypes voor SQL
        df['Altitude'] = pd.to_numeric(df['Altitude'], errors='coerce')
        df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
        df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

        df['LocationKey'] = df['LocationKey'].astype("Int64")
        
        return df

    except Exception as e:
        print(f"Er ging iets mis bij het downloaden van de stations: {e}")
        return None

In [8]:
df = download_aws_stations()

                FID  code              the_geom      name  \
0  aws_station.6414  6414  POINT (50.904 3.122)    BEITEM   
1  aws_station.6438  6438  POINT (51.325 4.364)  STABROEK   
2  aws_station.6464  6464  POINT (51.221 5.027)     RETIE   
3  aws_station.6447  6447  POINT (50.797 4.358)     UCCLE   
4  aws_station.6434  6434   POINT (50.98 3.816)     MELLE   

            date_begin  date_end  altitude  
0  2003-07-26T00:10:00       NaN      24.8  
1  2012-08-05T00:00:00       NaN       4.0  
2  2002-01-23T00:00:00       NaN      21.5  
3  2003-12-01T00:00:00       NaN     100.6  
4  2002-02-22T15:50:00       NaN      15.0  
BEITEM 8800 <class 'str'>
postcode gevonden in Vlaanderen
      Postcode Plaatsnaam Deelgemeente Hoofdgemeente        Provincie
2402      8800    Beveren           Ja     ROESELARE  WEST-VLAANDEREN
2403      8800     Oekene           Ja     ROESELARE  WEST-VLAANDEREN
2404      8800  Roeselare         Neen     ROESELARE  WEST-VLAANDEREN
2405      8800    Rumbeke